# Generic RAG Template — interactive demo

This is a Kaggle-runnable demo of `rag_pipeline.py`, a **generic** RAG
pipeline: point it at any folder of documents (.txt/.md/.csv/.pdf/.docx) and
get a grounded Q&A system. Unlike `OntologyRAG_Q_Kaggle.ipynb` (built
specifically for the Quran/Tafsir dataset), this pipeline makes no
assumptions about domain or language -- it's meant to be reused as-is for a
new client's documents in a consulting/freelance engagement.

## How to run on Kaggle
1. **New Notebook** -> Settings -> **Accelerator: GPU T4**, **Internet: On**
2. Upload this notebook AND `rag_pipeline.py` (Add Data / Upload, or paste
   `rag_pipeline.py`'s contents into a Kaggle Dataset/Utility script) so it's
   importable.
3. *(Optional)* Add `OPENAI_API_KEY` as a Kaggle secret for GPT-quality
   answers -- otherwise this uses a free local LLM.
4. Run All. Then edit the "Try it on your own documents" cell to point at a
   real client folder and ask your own questions.


In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers faiss-cpu pypdf python-docx


## 2. Make `rag_pipeline.py` importable

On Kaggle, upload `rag_pipeline.py` alongside this notebook (or as a Kaggle
Utility Script / Dataset) and adjust the path below if needed.


In [ ]:
import sys, os

# Adjust this path to wherever you uploaded rag_pipeline.py on Kaggle
CANDIDATE_PATHS = [".", "/kaggle/working", "/kaggle/input"]
for p in CANDIDATE_PATHS:
    if os.path.exists(os.path.join(p, "rag_pipeline.py")):
        sys.path.insert(0, p)
        break

from rag_pipeline import RAGPipeline
print("rag_pipeline loaded OK")


## 3. Create a small demo document set

Replace this cell with your own client's documents (or mount a Kaggle
Dataset containing them) -- this is just a stand-in so the notebook runs
end-to-end out of the box.


In [ ]:
import os

os.makedirs("demo_docs", exist_ok=True)

with open("demo_docs/refund_policy.txt", "w") as f:
    f.write("""
Our refund policy allows customers to request a full refund within 30 days
of purchase. After 30 days, only store credit is issued. Refunds are
processed within 5 business days of approval. Shipping costs are
non-refundable unless the item arrived defective or damaged.
""")

with open("demo_docs/support_faq.md", "w") as f:
    f.write("""
# Support FAQ

## How do I reset my password?
Go to Account Settings and click "Reset Password". A reset link will be
emailed to you within a few minutes.

## What are your support hours?
Support is available Monday to Friday, 9am to 6pm EST. Emails sent outside
these hours are answered the next business day.

## How do I cancel my subscription?
Go to Billing -> Manage Subscription -> Cancel. Cancellation takes effect
at the end of the current billing period; no partial refunds are issued for
the remaining days.
""")

print("Demo documents created")


## 4. Build the index

In [ ]:
rag = RAGPipeline()  # free local LLM by default; pass openai_api_key=... for GPT

try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("OPENAI_API_KEY")
    if key:
        rag = RAGPipeline(openai_api_key=key)
        print("Using OpenAI GPT for generation")
except Exception:
    print("Using free local LLM for generation (Qwen2.5-1.5B-Instruct)")

rag.build_index("demo_docs", chunk_size=60, overlap=10)
print(f"Indexed {len(rag._chunks)} chunks from demo_docs/")


## 5. Ask questions

In [ ]:
for question in [
    "How long do I have to request a refund?",
    "How do I reset my password?",
    "Can I get a partial refund if I cancel my subscription mid-month?",
]:
    answer, sources = rag.answer(question, top_k=2)
    print("Q:", question)
    print("A:", answer)
    print("Sources:", sorted({s.source for s in sources}))
    print()


## 6. Try it on your own documents

This is the cell to edit for a real engagement: point `build_index` at a
folder containing a client's actual documents (upload them as a Kaggle
Dataset, or `/kaggle/working/their_docs` if you copied them in), then ask
real questions from their domain.


In [ ]:
# rag_client = RAGPipeline()
# rag_client.build_index("/kaggle/input/client-documents", chunk_size=200, overlap=40)
#
# answer, sources = rag_client.answer("Your real question here")
# print(answer)
# print([s.source for s in sources])


## Notes

- **Persisting the index**: for a real (larger) document set, call
  `rag.save_index("my_index")` once, then `rag.load_index("my_index")` on
  later runs instead of re-embedding everything from scratch.
- **Tuning retrieval**: if answers are missing context, increase
  `chunk_size`; if answers seem to pull in irrelevant text, decrease it or
  lower `top_k`.
- **Evaluation**: before showing this to a client as "production-ready,"
  collect a handful of real question/answer pairs from their domain and
  check the generated answers against them -- see
  `../ontologyrag_q_kaggle/OntologyRAG_Q_Kaggle.ipynb` for a worked example
  of token-F1 + BERTScore evaluation you can adapt here.
